# 第140章: 予測のための表現学習 — ピクセルから特徴空間へ

## Representation Learning for Prediction: From Pixels to Feature Space

---

### このノートブックの位置づけ

World Models シリーズの一部として、**なぜピクセル空間ではなく特徴空間（潜在空間）で予測を行うべきか** を実験を通じて理解します。JEPA や Dreamer などの最先端モデルが採用する「潜在空間予測」の根拠を、対照学習とt-SNE可視化を用いて体感します。

### 学習目標

1. **ピクセル空間 vs 特徴空間** の予測がどう異なるかを理解する
2. **対照学習（Contrastive Learning）** と InfoNCE 損失の仕組みを実装する
3. **表現空間の t-SNE 可視化** で学習された表現の質を確認する
4. **線形プローブ評価** で表現の有用性を定量的に測定する

### 前提知識

- Notebook 36: PyTorch によるニューラルネットワーク構築
- Notebook 44: CLIP と対照学習の基礎概念
- Notebook 95: Vision Transformer (ViT) の仕組み

### メタ情報

| 項目 | 内容 |
|------|------|
| 難易度 | ★★★☆☆ |
| 所要時間 | 120〜150分 |
| 使用ライブラリ | PyTorch, NumPy, Matplotlib, scikit-learn |
| データセット | FashionMNIST（自動ダウンロード） |

---

## 目次

1. [環境セットアップ](#1-環境セットアップ)
2. [なぜ表現学習が必要か？](#2-なぜ表現学習が必要か)
3. [対照学習 (Contrastive Learning) の基礎](#3-対照学習-contrastive-learning-の基礎)
4. [SimpleEncoder の実装](#4-simpleencoder-の実装)
5. [FashionMNIST での対照学習訓練](#5-fashionmnist-での対照学習訓練)
6. [表現空間の t-SNE 可視化](#6-表現空間の-t-sne-可視化)
7. [線形プローブ評価 (Linear Probe)](#7-線形プローブ評価-linear-probe)
8. [ピクセル予測 vs 特徴空間予測](#8-ピクセル予測-vs-特徴空間予測)
9. [まとめ・よくある間違い・確認クイズ](#9-まとめよくある間違い確認クイズ)

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# 環境セットアップ
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset

import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定（環境に応じて調整）
import japanize_matplotlib # 日本語化
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 再現性のためのシード
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("環境セットアップ完了")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

---

## 2. なぜ表現学習が必要か？

### 2.1 ピクセル空間の問題点

画像をそのまま（ピクセルとして）扱う場合、以下の問題が生じます：

1. **次元が高すぎる**: 28x28 のグレースケール画像でも 784 次元。実用的な画像（例: 256x256x3）では約 20 万次元
2. **意味的に無関係な情報を含む**: 背景の微妙なノイズ、照明条件の違いなど
3. **距離が意味を反映しない**: ピクセル空間でのユークリッド距離は、意味的な類似度と一致しない

### 2.2 特徴空間の利点

良い **表現（representation）** を学習すると：

- **低次元**: 数百次元に圧縮（例: 128次元）
- **意味的構造**: 意味が近い画像は特徴空間でも近い
- **タスク非依存**: 一度学習した表現が様々な下流タスクで使える

### World Models との接続

JEPA（Joint Embedding Predictive Architecture）や Dreamer のような世界モデルは、**ピクセル空間ではなく潜在空間で未来を予測** します。これにより：

- 意味的に重要な変化のみを予測できる
- 計算コストが大幅に削減される
- 不必要な詳細（ノイズ）を無視できる

In [ ]:
# ============================================================
# 2.3 FashionMNIST データの読み込み
# ============================================================

# 基本の変換（正規化のみ）
basic_transform = transforms.Compose([
    transforms.ToTensor(),
])

# FashionMNIST の読み込み
train_dataset_full = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=basic_transform
)
test_dataset_full = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=basic_transform
)

# 教育用に小規模サブセットを使用（訓練5000枚、テスト1000枚）
train_indices = np.random.choice(len(train_dataset_full), 5000, replace=False)
test_indices = np.random.choice(len(test_dataset_full), 1000, replace=False)

train_subset = Subset(train_dataset_full, train_indices)
test_subset = Subset(test_dataset_full, test_indices)

# クラス名（日本語）
class_names = [
    'Tシャツ', 'ズボン', 'プルオーバー', 'ドレス', 'コート',
    'サンダル', 'シャツ', 'スニーカー', 'バッグ', 'アンクルブーツ'
]

print(f"訓練データ: {len(train_subset)} 枚")
print(f"テストデータ: {len(test_subset)} 枚")
print(f"クラス数: {len(class_names)}")
print(f"画像サイズ: {train_dataset_full[0][0].shape}")

In [ ]:
# ============================================================
# 2.4 ピクセル空間での距離 vs 意味的距離の可視化
# ============================================================

def find_images_by_class(dataset, target_class, n=5):
    """指定クラスの画像をn枚見つける"""
    images = []
    for img, label in dataset:
        if label == target_class:
            images.append(img)
        if len(images) >= n:
            break
    return images

# 同じクラス（Tシャツ）から2枚、異なるクラス（サンダル）から1枚を取得
tshirts = find_images_by_class(train_dataset_full, 0, n=2)
sandals = find_images_by_class(train_dataset_full, 5, n=1)

img_a = tshirts[0]  # Tシャツ 1
img_b = tshirts[1]  # Tシャツ 2
img_c = sandals[0]  # サンダル

# ピクセル空間でのユークリッド距離を計算
dist_ab = torch.norm(img_a - img_b).item()  # 同クラス間
dist_ac = torch.norm(img_a - img_c).item()  # 異クラス間
dist_bc = torch.norm(img_b - img_c).item()  # 異クラス間

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# 3枚の画像を表示
axes[0].imshow(img_a.squeeze(), cmap='gray')
axes[0].set_title('画像A: Tシャツ 1', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_b.squeeze(), cmap='gray')
axes[1].set_title('画像B: Tシャツ 2', fontsize=12)
axes[1].axis('off')

axes[2].imshow(img_c.squeeze(), cmap='gray')
axes[2].set_title('画像C: サンダル', fontsize=12)
axes[2].axis('off')

# 差分画像を表示
diff_ab = torch.abs(img_a - img_b).squeeze()
axes[3].imshow(diff_ab, cmap='hot')
axes[3].set_title('|A - B| の差分', fontsize=12)
axes[3].axis('off')

plt.suptitle('ピクセル空間での距離は意味的類似度を反映しない', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"ピクセル空間のユークリッド距離:")
print(f"  d(Tシャツ1, Tシャツ2) = {dist_ab:.2f}  ← 同じカテゴリなのに…")
print(f"  d(Tシャツ1, サンダル)  = {dist_ac:.2f}  ← 違うカテゴリ")
print(f"  d(Tシャツ2, サンダル)  = {dist_bc:.2f}  ← 違うカテゴリ")
print()
print("【問題点】同じ『Tシャツ』同士の距離が、")
print("異なるカテゴリ間の距離と大差ない（場合によっては大きい）。")
print("→ ピクセル距離は意味的な類似度を捉えられていない！")

In [ ]:
# ============================================================
# 2.3b FashionMNIST のサンプル表示
# ============================================================

fig, axes = plt.subplots(2, 10, figsize=(16, 4))

# 各クラスから2枚ずつ表示
for class_idx in range(10):
    imgs = find_images_by_class(train_dataset_full, class_idx, n=2)
    for row in range(2):
        axes[row, class_idx].imshow(imgs[row].squeeze(), cmap='gray')
        axes[row, class_idx].axis('off')
        if row == 0:
            axes[row, class_idx].set_title(class_names[class_idx], fontsize=9)

plt.suptitle('FashionMNIST: 各クラスのサンプル画像', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("10クラスの衣類画像データセット")
print("同じクラス内でもデザイン・形状が多様 → ピクセル距離では捉えにくい")

In [ ]:
# ============================================================
# 2.5 ピクセル空間距離の統計的分析
# ============================================================

def compute_pairwise_distances(dataset, n_samples=500):
    """同クラスペアと異クラスペアのピクセル距離を計算する"""
    images = []
    labels = []
    for i in range(min(n_samples, len(dataset))):
        img, label = dataset[i]
        images.append(img.flatten())
        labels.append(label)

    images = torch.stack(images)
    labels = torch.tensor(labels)

    same_class_dists = []
    diff_class_dists = []

    # ランダムに1000ペアをサンプリング
    for _ in range(1000):
        i, j = np.random.choice(len(images), 2, replace=False)
        dist = torch.norm(images[i] - images[j]).item()
        if labels[i] == labels[j]:
            same_class_dists.append(dist)
        else:
            diff_class_dists.append(dist)

    return same_class_dists, diff_class_dists

same_dists, diff_dists = compute_pairwise_distances(train_dataset_full)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(same_dists, bins=30, alpha=0.6, label='同クラスペア', color='blue', density=True)
ax.hist(diff_dists, bins=30, alpha=0.6, label='異クラスペア', color='red', density=True)
ax.set_xlabel('ピクセル空間のユークリッド距離', fontsize=12)
ax.set_ylabel('密度', fontsize=12)
ax.set_title('ピクセル空間: 同クラス vs 異クラスの距離分布', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"同クラスペアの平均距離: {np.mean(same_dists):.2f} (std: {np.std(same_dists):.2f})")
print(f"異クラスペアの平均距離: {np.mean(diff_dists):.2f} (std: {np.std(diff_dists):.2f})")
print()
print("【観察】2つの分布が大きく重なっている = ピクセル距離で分類するのは困難")
print("→ 表現学習で、この重なりを解消したい！")

---

## 3. 対照学習 (Contrastive Learning) の基礎

### 3.1 対照学習とは

対照学習は、**ラベルなしデータ** から有用な表現を学習する自己教師あり学習の一種です。核心的なアイデアは：

> 「意味的に近い」サンプル同士を特徴空間で近づけ、
> 「意味的に遠い」サンプル同士を遠ざける

### 3.2 正例ペアと負例ペアの作り方

ラベルがない場合、どうやって「意味的に近い」を定義するか？

**データ拡張（Data Augmentation）** を活用します：

- **正例ペア（positive pair）**: 同じ画像に異なる拡張を適用した2つのビュー
- **負例ペア（negative pair）**: 異なる画像から作られたペア

直感：同じ画像を少し変形しても「意味」は変わらない → 特徴空間で近くあるべき

### 3.3 InfoNCE 損失関数

対照学習で最も広く使われる損失関数が **InfoNCE** です：

$$
\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbf{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k) / \tau)}
$$

ここで：
- $z_i, z_j$: 正例ペアの特徴ベクトル
- $\text{sim}(u, v) = \frac{u \cdot v}{\|u\| \|v\|}$: コサイン類似度
- $\tau$: 温度パラメータ（temperature）
- $N$: バッチサイズ

**直感的な理解**: InfoNCE は「正例ペアの類似度を、全てのペアの類似度の中で際立たせる」ソフトマックス分類問題です。

In [ ]:
# ============================================================
# 3.4 温度パラメータの効果を可視化
# ============================================================

def visualize_temperature_effect():
    """温度パラメータ τ が確率分布に与える影響を可視化する"""
    # 仮の類似度ベクトル（正例が最も高い）
    similarities = torch.tensor([0.9, 0.5, 0.3, 0.1, -0.1, -0.3])
    labels = ['正例', '負例1', '負例2', '負例3', '負例4', '負例5']

    temperatures = [0.1, 0.5, 1.0, 2.0]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    for ax, tau in zip(axes, temperatures):
        # ソフトマックス確率を計算
        probs = F.softmax(similarities / tau, dim=0).numpy()

        colors = ['green'] + ['salmon'] * 5
        bars = ax.bar(labels, probs, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_title(f'$\\tau$ = {tau}', fontsize=13)
        ax.set_ylabel('確率' if ax == axes[0] else '')
        ax.set_ylim(0, 1.0)
        ax.tick_params(axis='x', rotation=45)

        # 正例の確率を表示
        ax.text(0, probs[0] + 0.02, f'{probs[0]:.3f}', ha='center', fontsize=9,
                fontweight='bold')

    plt.suptitle('温度パラメータ τ の効果：低い τ ほど確率が鋭くなる', fontsize=14, y=1.05)
    plt.tight_layout()
    plt.show()

    print("【温度パラメータの役割】")
    print("・τ が小さい → 確率分布が鋭い → 正例 vs 負例の区別が厳密")
    print("・τ が大きい → 確率分布が平坦 → よりソフトな区別")
    print("・実用上は τ = 0.07〜0.5 程度がよく使われる")

visualize_temperature_effect()

### 3.4b 対照学習のパイプライン概念図

対照学習の全体的な流れを整理します：

```
入力画像 x
  ├─ 拡張1 → x_i ─→ Encoder ─→ h_i ─→ Projection Head ─→ z_i ─┐
  │                                                                ├─ InfoNCE Loss
  └─ 拡張2 → x_j ─→ Encoder ─→ h_j ─→ Projection Head ─→ z_j ─┘
              (共有重み)        (共有重み)        (共有重み)

  学習時: z_i, z_j を使って InfoNCE 損失を計算
  評価時: h_i（エンコーダ出力）を表現として使用
```

**重要**: Encoder と Projection Head は2つのビュー間で **重み共有** されます。

In [ ]:
# ============================================================
# 3.5 ContrastiveLoss（InfoNCE）の実装
# ============================================================

class ContrastiveLoss(nn.Module):
    """InfoNCE 対照損失

    同じ画像の異なる拡張（正例ペア）の特徴を近づけ、
    異なる画像（負例ペア）の特徴を遠ざける。

    Parameters
    ----------
    temperature : float
        温度パラメータ。小さいほど鋭い確率分布になる。
    """

    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_i, z_j):
        """InfoNCE損失を計算する

        Parameters
        ----------
        z_i : torch.Tensor, shape (batch_size, embed_dim)
            拡張ビュー1の特徴ベクトル
        z_j : torch.Tensor, shape (batch_size, embed_dim)
            拡張ビュー2の特徴ベクトル

        Returns
        -------
        loss : torch.Tensor
            InfoNCE損失のスカラー値
        """
        batch_size = z_i.shape[0]

        # L2正規化（コサイン類似度のため）
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)

        # 全ペアを結合: [z_i; z_j] → shape (2*batch_size, embed_dim)
        z = torch.cat([z_i, z_j], dim=0)

        # 全ペア間のコサイン類似度行列: shape (2N, 2N)
        sim_matrix = torch.mm(z, z.t()) / self.temperature

        # 自分自身との類似度を除外（対角要素をマスク）
        mask = torch.eye(2 * batch_size, device=z.device).bool()
        sim_matrix = sim_matrix.masked_fill(mask, float('-inf'))

        # 正例ペアのインデックスを作成
        # z_i[k] の正例は z_j[k]（インデックス k+N）、逆も同様
        positive_indices = torch.cat([
            torch.arange(batch_size, 2 * batch_size),  # z_i → z_j
            torch.arange(0, batch_size)                 # z_j → z_i
        ]).to(z.device)

        # クロスエントロピー損失（InfoNCEはソフトマックス分類と同等）
        loss = F.cross_entropy(sim_matrix, positive_indices)

        return loss


# 動作確認
loss_fn = ContrastiveLoss(temperature=0.5)

# ダミーデータでテスト
dummy_z_i = torch.randn(4, 128)  # バッチ4、128次元
dummy_z_j = torch.randn(4, 128)
dummy_loss = loss_fn(dummy_z_i, dummy_z_j)
print(f"ダミーデータでの InfoNCE 損失: {dummy_loss.item():.4f}")

# 正例が完全に一致する場合（損失が小さくなるはず）
identical_loss = loss_fn(dummy_z_i, dummy_z_i + 0.01 * torch.randn_like(dummy_z_i))
print(f"ほぼ同一ペアでの損失: {identical_loss.item():.4f}")
print(f"→ 正例が近いほど損失が小さい: {identical_loss.item() < dummy_loss.item()}")

---

## 4. SimpleEncoder の実装

### 4.1 エンコーダの設計方針

対照学習で使うエンコーダは、画像を **低次元の特徴ベクトル（embedding）** に変換します。

本ノートブックでは教育目的のため、シンプルな3層CNNを使用します：

```
入力画像 (1, 28, 28)
  → Conv2d(32) + ReLU + MaxPool → (32, 14, 14)
  → Conv2d(64) + ReLU + MaxPool → (64, 7, 7)
  → Conv2d(128) + ReLU + AdaptiveAvgPool → (128, 1, 1)
  → Flatten → 128次元の埋め込み
```

### 4.2 プロジェクションヘッド

対照学習では、エンコーダの出力をさらに **プロジェクションヘッド** で変換してから損失を計算します（SimCLR の知見）。

- 学習時: エンコーダ出力 → プロジェクションヘッド → InfoNCE 損失
- 評価時: エンコーダ出力のみを表現として使用（プロジェクションヘッドは捨てる）

In [ ]:
# ============================================================
# 4.3 SimpleEncoder の実装
# ============================================================

class SimpleEncoder(nn.Module):
    """対照学習用のシンプルなCNNエンコーダ

    3層の畳み込み層で画像を128次元の特徴ベクトルに変換する。
    プロジェクションヘッドは対照学習の損失計算用。

    Parameters
    ----------
    embed_dim : int
        エンコーダ出力の埋め込み次元数（デフォルト: 128）
    projection_dim : int
        プロジェクションヘッド出力の次元数（デフォルト: 64）
    """

    def __init__(self, embed_dim=128, projection_dim=64):
        super().__init__()

        # エンコーダ: 3層の畳み込みネットワーク
        self.encoder = nn.Sequential(
            # 第1層: (1, 28, 28) → (32, 14, 14)
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 第2層: (32, 14, 14) → (64, 7, 7)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 第3層: (64, 7, 7) → (128, 1, 1)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # グローバル平均プーリング
        )

        # 埋め込み層: 128 → embed_dim
        self.fc = nn.Linear(128, embed_dim)

        # プロジェクションヘッド（対照学習の損失計算用）
        self.projection_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, projection_dim),
        )

    def get_embedding(self, x):
        """エンコーダの埋め込みのみを返す（評価時に使用）"""
        h = self.encoder(x)
        h = h.view(h.size(0), -1)  # Flatten
        embedding = self.fc(h)
        return embedding

    def forward(self, x):
        """埋め込み + プロジェクションを返す（学習時に使用）"""
        embedding = self.get_embedding(x)
        projection = self.projection_head(embedding)
        return embedding, projection


# モデルの確認
model = SimpleEncoder(embed_dim=128, projection_dim=64).to(device)

# ダミー入力で形状を確認
dummy_input = torch.randn(2, 1, 28, 28).to(device)
embedding, projection = model(dummy_input)

print("SimpleEncoder のアーキテクチャ:")
print(f"  入力形状:         {dummy_input.shape}")
print(f"  埋め込み形状:     {embedding.shape}")
print(f"  プロジェクション形状: {projection.shape}")
print(f"\n総パラメータ数: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================
# 4.4 データ拡張パイプライン
# ============================================================

class ContrastiveAugmentation:
    """対照学習用のデータ拡張

    同じ画像に対して2つの異なる拡張ビューを生成する。
    FashionMNIST（グレースケール28x28）向けのシンプルな拡張。
    """

    def __init__(self):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(28, scale=(0.8, 1.0)),  # ランダムクロップ
            transforms.RandomHorizontalFlip(p=0.5),               # 水平反転
            transforms.RandomApply([
                transforms.RandomAffine(
                    degrees=10,          # 回転 ±10度
                    translate=(0.1, 0.1), # 平行移動
                )
            ], p=0.5),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),  # ランダム消去
        ])

    def __call__(self, x):
        """2つの異なる拡張ビューを返す"""
        view1 = self.transform(x)
        view2 = self.transform(x)
        return view1, view2


class ContrastiveDataset(Dataset):
    """対照学習用データセット

    各サンプルに対して2つの拡張ビューを返す。
    """

    def __init__(self, base_dataset, augmentation):
        self.base_dataset = base_dataset
        self.augmentation = augmentation

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]
        view1, view2 = self.augmentation(img)
        return view1, view2, label


# 拡張の可視化
augmentation = ContrastiveAugmentation()

fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for row in range(3):
    # 元の画像
    img, label = train_subset[row]
    axes[row, 0].imshow(img.squeeze(), cmap='gray')
    axes[row, 0].set_title(f'元画像: {class_names[label]}', fontsize=10)

    # 4つの異なる拡張を表示
    for col in range(1, 5):
        v1, v2 = augmentation(img)
        axes[row, col].imshow(v1.squeeze(), cmap='gray')
        axes[row, col].set_title(f'拡張 {col}', fontsize=10)

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('データ拡張の例：同じ画像から多様なビューを生成', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("【データ拡張の役割】")
print("・拡張を変えても『同じ服のカテゴリ』であるという不変性を学ばせる")
print("・拡張が強すぎると情報が失われ、弱すぎるとショートカットを学習する")

---

## 5. FashionMNIST での対照学習訓練

### 5.1 訓練ループの設計

対照学習の訓練ループ：

1. ミニバッチから各画像の2つの拡張ビューを作成
2. 両ビューをエンコーダに通してプロジェクションを取得
3. InfoNCE 損失を計算
4. 逆伝播でパラメータを更新

**注意**: ラベルは一切使用しません（自己教師あり学習）。

In [ ]:
# ============================================================
# 5.2 train_contrastive: 対照学習の訓練関数
# ============================================================

def train_contrastive(model, train_dataset, n_epochs=10, batch_size=128,
                      lr=1e-3, temperature=0.5, device='cpu'):
    """対照学習でエンコーダを訓練する

    Parameters
    ----------
    model : SimpleEncoder
        学習するエンコーダモデル
    train_dataset : Dataset
        訓練データセット（元のデータ、拡張はContrastiveDataset内で行う）
    n_epochs : int
        エポック数
    batch_size : int
        バッチサイズ
    lr : float
        学習率
    temperature : float
        InfoNCE の温度パラメータ
    device : str
        デバイス ('cpu' or 'cuda')

    Returns
    -------
    loss_history : list of float
        各エポックの平均損失
    """
    # 対照学習用データセットとデータローダー
    augmentation = ContrastiveAugmentation()
    contrastive_dataset = ContrastiveDataset(train_dataset, augmentation)
    dataloader = DataLoader(
        contrastive_dataset, batch_size=batch_size, shuffle=True, drop_last=True
    )

    # 損失関数とオプティマイザ
    criterion = ContrastiveLoss(temperature=temperature)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    loss_history = []

    for epoch in range(n_epochs):
        epoch_loss = 0.0
        n_batches = 0

        for view1, view2, _ in dataloader:
            # ラベル（_）は使用しない！
            view1 = view1.to(device)
            view2 = view2.to(device)

            # 順伝播：プロジェクションを取得
            _, proj1 = model(view1)
            _, proj2 = model(view2)

            # InfoNCE 損失の計算
            loss = criterion(proj1, proj2)

            # 逆伝播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        loss_history.append(avg_loss)

        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1:>2}/{n_epochs}]  Loss: {avg_loss:.4f}")

    return loss_history


print("train_contrastive 関数の定義完了")

In [ ]:
# ============================================================
# 5.3 対照学習の実行
# ============================================================

print("対照学習を開始します...")
print(f"データ: {len(train_subset)} 枚, デバイス: {device}")
print("="*50)

# モデルの初期化
contrastive_model = SimpleEncoder(embed_dim=128, projection_dim=64).to(device)

# 訓練実行（教育用のため、小規模・短エポック）
loss_history = train_contrastive(
    model=contrastive_model,
    train_dataset=train_subset,
    n_epochs=15,
    batch_size=128,
    lr=1e-3,
    temperature=0.5,
    device=device,
)

print("="*50)
print("対照学習が完了しました！")

In [ ]:
# ============================================================
# 5.4 損失曲線の可視化
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(loss_history) + 1), loss_history, 'o-', linewidth=2, markersize=6,
        color='dodgerblue')
ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('InfoNCE 損失', fontsize=12)
ax.set_title('対照学習の損失曲線', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, len(loss_history) + 1))

# 理論的下限値の参照線
# InfoNCE の下限 ≈ -log(1/N) = log(N)（N = 2*batch_size - 1）
# ただし実際にはここまで下がらないことが多い
ax.axhline(y=loss_history[-1], color='red', linestyle='--', alpha=0.5,
           label=f'最終損失: {loss_history[-1]:.4f}')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"初期損失: {loss_history[0]:.4f}")
print(f"最終損失: {loss_history[-1]:.4f}")
print(f"損失の減少率: {(1 - loss_history[-1]/loss_history[0])*100:.1f}%")

---

## 6. 表現空間の t-SNE 可視化

### 6.1 t-SNE とは

t-SNE（t-distributed Stochastic Neighbor Embedding）は、高次元データを2次元に可視化するための手法です。

- 高次元空間での近傍関係を低次元空間でも保存しようとする
- 同じクラスのデータが固まって表示されれば、表現がクラス構造を捉えていることを示す

### 6.2 比較実験

以下の2つの埋め込みを比較します：

1. **ランダム初期化のエンコーダ** の出力（学習前）
2. **対照学習で訓練したエンコーダ** の出力（学習後）

良い表現が学習されていれば、(2) ではクラスごとに明確なクラスタが形成されるはずです。

In [ ]:
# ============================================================
# 6.3 埋め込みの抽出関数
# ============================================================

@torch.no_grad()
def extract_embeddings(model, dataset, device='cpu', max_samples=1000):
    """データセットからエンコーダの埋め込みを抽出する

    Parameters
    ----------
    model : SimpleEncoder
        エンコーダモデル
    dataset : Dataset
        データセット
    device : str
        デバイス
    max_samples : int
        最大サンプル数

    Returns
    -------
    embeddings : numpy.ndarray, shape (n_samples, embed_dim)
    labels : numpy.ndarray, shape (n_samples,)
    """
    model.eval()
    dataloader = DataLoader(dataset, batch_size=256, shuffle=False)

    all_embeddings = []
    all_labels = []
    n_collected = 0

    for images, labels in dataloader:
        if n_collected >= max_samples:
            break

        images = images.to(device)
        embeddings = model.get_embedding(images)
        all_embeddings.append(embeddings.cpu().numpy())
        all_labels.append(labels.numpy())
        n_collected += len(images)

    all_embeddings = np.concatenate(all_embeddings, axis=0)[:max_samples]
    all_labels = np.concatenate(all_labels, axis=0)[:max_samples]

    return all_embeddings, all_labels


print("extract_embeddings 関数の定義完了")

In [ ]:
# ============================================================
# 6.4 ランダム初期化 vs 対照学習の埋め込みを抽出
# ============================================================

# ランダム初期化のエンコーダ（学習前の対照として）
random_model = SimpleEncoder(embed_dim=128, projection_dim=64).to(device)

print("埋め込みを抽出中...")

# テストデータから埋め込みを抽出
random_embeddings, test_labels = extract_embeddings(
    random_model, test_subset, device=device, max_samples=1000
)
trained_embeddings, _ = extract_embeddings(
    contrastive_model, test_subset, device=device, max_samples=1000
)

print(f"埋め込み形状: {random_embeddings.shape}")
print(f"ラベル形状: {test_labels.shape}")
print(f"クラスの分布: {np.bincount(test_labels)}")

In [ ]:
# ============================================================
# 6.5 t-SNE で2次元に可視化
# ============================================================

print("t-SNE を実行中（少し時間がかかります）...")

# t-SNE の実行
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
random_tsne = tsne.fit_transform(random_embeddings)

tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
trained_tsne = tsne.fit_transform(trained_embeddings)

print("t-SNE 完了！")

In [ ]:
# ============================================================
# 6.6 t-SNE 結果の比較可視化
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# カラーマップの設定
cmap = plt.cm.tab10
colors = [cmap(i) for i in range(10)]

# 左: ランダム初期化の埋め込み
for i in range(10):
    mask = test_labels == i
    axes[0].scatter(
        random_tsne[mask, 0], random_tsne[mask, 1],
        c=[colors[i]], label=class_names[i], s=15, alpha=0.6
    )
axes[0].set_title('ランダム初期化（学習前）', fontsize=14)
axes[0].set_xlabel('t-SNE 次元 1')
axes[0].set_ylabel('t-SNE 次元 2')
axes[0].legend(fontsize=8, loc='best', ncol=2, markerscale=2)

# 右: 対照学習後の埋め込み
for i in range(10):
    mask = test_labels == i
    axes[1].scatter(
        trained_tsne[mask, 0], trained_tsne[mask, 1],
        c=[colors[i]], label=class_names[i], s=15, alpha=0.6
    )
axes[1].set_title('対照学習後', fontsize=14)
axes[1].set_xlabel('t-SNE 次元 1')
axes[1].set_ylabel('t-SNE 次元 2')
axes[1].legend(fontsize=8, loc='best', ncol=2, markerscale=2)

plt.suptitle('t-SNE 可視化: 表現の質の比較', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("・左（学習前）: 色がバラバラに混ざっている → クラス構造が捉えられていない")
print("・右（学習後）: 同じ色が固まっている → 対照学習が意味的構造を学習できた")
print("・特に、靴系（サンダル、スニーカー、アンクルブーツ）が近くに配置されるなど、")
print("  カテゴリ間の意味的関係も反映されている可能性がある")

In [ ]:
# ============================================================
# 6.6b コサイン類似度行列の可視化
# ============================================================

def plot_similarity_matrix(embeddings, labels, class_names, title, n_per_class=5):
    """クラスごとにソートした埋め込みのコサイン類似度行列を可視化する"""
    # 各クラスからn_per_class枚ずつ選択
    selected_indices = []
    selected_labels = []
    for c in range(10):
        class_indices = np.where(labels == c)[0][:n_per_class]
        selected_indices.extend(class_indices)
        selected_labels.extend([c] * len(class_indices))

    selected_emb = embeddings[selected_indices]
    selected_labels = np.array(selected_labels)

    # L2正規化してコサイン類似度を計算
    norms = np.linalg.norm(selected_emb, axis=1, keepdims=True) + 1e-8
    normed = selected_emb / norms
    sim_matrix = normed @ normed.T

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(sim_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax, label='コサイン類似度')

    # クラス境界線を描画
    for i in range(1, 10):
        pos = i * n_per_class
        ax.axhline(y=pos - 0.5, color='black', linewidth=0.5)
        ax.axvline(x=pos - 0.5, color='black', linewidth=0.5)

    # クラスラベル
    tick_positions = [i * n_per_class + n_per_class // 2 for i in range(10)]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([class_names[i][:3] for i in range(10)], fontsize=8, rotation=45)
    ax.set_yticks(tick_positions)
    ax.set_yticklabels([class_names[i][:3] for i in range(10)], fontsize=8)
    ax.set_title(title, fontsize=13)

    return fig

fig1 = plot_similarity_matrix(random_embeddings, test_labels, class_names,
                              'ランダム初期化: コサイン類似度行列')
plt.tight_layout()
plt.show()

fig2 = plot_similarity_matrix(trained_embeddings, test_labels, class_names,
                              '対照学習後: コサイン類似度行列')
plt.tight_layout()
plt.show()

print("【コサイン類似度行列の読み方】")
print("・対角ブロック（同クラス同士）が赤い = 同クラスの類似度が高い")
print("・非対角ブロック（異クラス同士）が青い = 異クラスの類似度が低い")
print("・対照学習後は、対角ブロックがより明確な赤、非対角が青になるはず")

In [ ]:
# ============================================================
# 6.7 特徴空間での距離分布の改善を確認
# ============================================================

def compute_embedding_distances(embeddings, labels, n_pairs=1000):
    """埋め込み空間での同クラス/異クラスの距離を計算する"""
    same_dists = []
    diff_dists = []

    for _ in range(n_pairs):
        i, j = np.random.choice(len(embeddings), 2, replace=False)
        dist = np.linalg.norm(embeddings[i] - embeddings[j])
        if labels[i] == labels[j]:
            same_dists.append(dist)
        else:
            diff_dists.append(dist)

    return same_dists, diff_dists

# ランダム初期化 vs 対照学習後
rand_same, rand_diff = compute_embedding_distances(random_embeddings, test_labels)
trained_same, trained_diff = compute_embedding_distances(trained_embeddings, test_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ランダム初期化
axes[0].hist(rand_same, bins=30, alpha=0.6, label='同クラス', color='blue', density=True)
axes[0].hist(rand_diff, bins=30, alpha=0.6, label='異クラス', color='red', density=True)
axes[0].set_title('ランダム初期化: 特徴空間の距離', fontsize=12)
axes[0].set_xlabel('ユークリッド距離')
axes[0].set_ylabel('密度')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 対照学習後
axes[1].hist(trained_same, bins=30, alpha=0.6, label='同クラス', color='blue', density=True)
axes[1].hist(trained_diff, bins=30, alpha=0.6, label='異クラス', color='red', density=True)
axes[1].set_title('対照学習後: 特徴空間の距離', fontsize=12)
axes[1].set_xlabel('ユークリッド距離')
axes[1].set_ylabel('密度')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('特徴空間での距離分布: 学習前 vs 学習後', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("【改善の定量評価】")
print(f"ランダム初期化 - 同クラス平均: {np.mean(rand_same):.2f}, 異クラス平均: {np.mean(rand_diff):.2f}")
print(f"対照学習後     - 同クラス平均: {np.mean(trained_same):.2f}, 異クラス平均: {np.mean(trained_diff):.2f}")

# 分離度の指標: (異クラス平均 - 同クラス平均) / (異クラス標準偏差 + 同クラス標準偏差)
rand_sep = (np.mean(rand_diff) - np.mean(rand_same)) / (np.std(rand_diff) + np.std(rand_same))
trained_sep = (np.mean(trained_diff) - np.mean(trained_same)) / (np.std(trained_diff) + np.std(trained_same))
print(f"\n分離度スコア: ランダム={rand_sep:.3f}, 対照学習後={trained_sep:.3f}")
print(f"→ 対照学習により、同クラスと異クラスの距離分布の分離が改善された")

---

## 7. 線形プローブ評価 (Linear Probe)

### 7.1 線形プローブとは

表現の質を定量的に評価する標準的な方法が **線形プローブ（Linear Probe）** です。

手順：
1. 学習済みエンコーダの重みを **凍結**（freeze）する
2. エンコーダの出力（埋め込み）の上に **線形分類器のみ** を学習する
3. 分類精度で表現の質を評価する

**直感**: もし埋め込みが良い表現であれば、線形分類器だけでも高い精度が出るはずです。

### 7.2 比較対象

| 表現 | 説明 |
|------|------|
| ランダム特徴 | ランダム初期化エンコーダの出力 |
| 対照学習特徴 | 対照学習で訓練したエンコーダの出力 |
| 教師あり特徴 | ラベル付きで教師あり学習したエンコーダの出力 |

### 7.2b なぜ線形プローブが表現の質の指標になるのか？

線形分類器は**非常に表現力が限られた**モデルです。線形分類器が高い精度を出すためには、入力特徴が**線形分離可能**（つまり、超平面で分けられる）でなければなりません。

```
悪い表現: クラスが入り組んでいる → 線形分類器では分離できない

  x o x        o: クラス1
  o x o        x: クラス2
  x o x        → 直線では分けられない！

良い表現: クラスが綺麗に分かれている → 線形分類器で十分

  o o o | x x x
  o o o | x x x    → 直線で分けられる！
```

したがって、線形プローブの精度が高い = 表現がクラス情報を**線形に分離可能な形で**エンコードしている、ということを意味します。

In [ ]:
# ============================================================
# 7.3 linear_probe: 線形プローブ評価関数
# ============================================================

def linear_probe(train_embeddings, train_labels, test_embeddings, test_labels):
    """線形プローブで表現の質を評価する

    エンコーダの埋め込み上にロジスティック回帰（線形分類器）を学習し、
    テストデータでの分類精度を返す。

    Parameters
    ----------
    train_embeddings : numpy.ndarray, shape (n_train, embed_dim)
        訓練データの埋め込み
    train_labels : numpy.ndarray, shape (n_train,)
        訓練データのラベル
    test_embeddings : numpy.ndarray, shape (n_test, embed_dim)
        テストデータの埋め込み
    test_labels : numpy.ndarray, shape (n_test,)
        テストデータのラベル

    Returns
    -------
    accuracy : float
        テストデータでの分類精度
    """
    # ロジスティック回帰（線形分類器）
    clf = LogisticRegression(
        max_iter=1000,
        solver='lbfgs',
        multi_class='multinomial',
        random_state=SEED,
    )

    # 学習
    clf.fit(train_embeddings, train_labels)

    # 評価
    predictions = clf.predict(test_embeddings)
    accuracy = accuracy_score(test_labels, predictions)

    return accuracy


print("linear_probe 関数の定義完了")

In [ ]:
# ============================================================
# 7.4 教師あり学習のベースラインを作成
# ============================================================

def train_supervised(model, train_dataset, n_epochs=15, batch_size=128,
                     lr=1e-3, device='cpu'):
    """教師あり学習（ラベル使用）でエンコーダを訓練する"""
    dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # 分類ヘッド
    classifier_head = nn.Linear(128, 10).to(device)

    # すべてのパラメータを最適化
    params = list(model.parameters()) + list(classifier_head.parameters())
    optimizer = optim.Adam(params, lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        n_batches = 0

        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            # 埋め込みを取得
            embeddings = model.get_embedding(images)
            logits = classifier_head(embeddings)

            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch [{epoch+1:>2}/{n_epochs}]  Loss: {epoch_loss/n_batches:.4f}")


# 教師あり学習モデルの訓練
print("教師あり学習のベースラインを訓練中...")
supervised_model = SimpleEncoder(embed_dim=128, projection_dim=64).to(device)
train_supervised(supervised_model, train_subset, n_epochs=15, device=device)
print("教師あり学習の訓練完了！")

In [ ]:
# ============================================================
# 7.5 線形プローブの実行と比較
# ============================================================

print("各モデルの埋め込みを抽出中...")

# 訓練データの埋め込み
random_train_emb, train_labels_arr = extract_embeddings(
    random_model, train_subset, device=device, max_samples=5000
)
contrastive_train_emb, _ = extract_embeddings(
    contrastive_model, train_subset, device=device, max_samples=5000
)
supervised_train_emb, _ = extract_embeddings(
    supervised_model, train_subset, device=device, max_samples=5000
)

# テストデータの埋め込み
random_test_emb, test_labels_arr = extract_embeddings(
    random_model, test_subset, device=device, max_samples=1000
)
contrastive_test_emb, _ = extract_embeddings(
    contrastive_model, test_subset, device=device, max_samples=1000
)
supervised_test_emb, _ = extract_embeddings(
    supervised_model, test_subset, device=device, max_samples=1000
)

print("\n線形プローブを実行中...")

# 線形プローブ
random_acc = linear_probe(random_train_emb, train_labels_arr,
                          random_test_emb, test_labels_arr)
contrastive_acc = linear_probe(contrastive_train_emb, train_labels_arr,
                               contrastive_test_emb, test_labels_arr)
supervised_acc = linear_probe(supervised_train_emb, train_labels_arr,
                              supervised_test_emb, test_labels_arr)

print("\n" + "="*50)
print("線形プローブ評価結果")
print("="*50)
print(f"  ランダム特徴:   {random_acc*100:.1f}%")
print(f"  対照学習特徴:   {contrastive_acc*100:.1f}%")
print(f"  教師あり特徴:   {supervised_acc*100:.1f}%")

In [ ]:
# ============================================================
# 7.6 結果の棒グラフ
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))

methods = ['ランダム特徴\n（学習なし）', '対照学習特徴\n（ラベルなし）', '教師あり特徴\n（ラベルあり）']
accuracies = [random_acc * 100, contrastive_acc * 100, supervised_acc * 100]
bar_colors = ['gray', 'dodgerblue', 'orange']

bars = ax.bar(methods, accuracies, color=bar_colors, edgecolor='black', linewidth=0.8, width=0.5)

# 値のラベル
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_ylabel('テスト精度 (%)', fontsize=12)
ax.set_title('線形プローブ評価: 表現の質の比較', fontsize=14)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("【考察】")
print(f"・ランダム特徴（{random_acc*100:.1f}%）: エンコーダ構造だけでもある程度の情報を抽出")
print(f"・対照学習特徴（{contrastive_acc*100:.1f}%）: ラベルなしでも意味的に有用な表現を学習")
print(f"・教師あり特徴（{supervised_acc*100:.1f}%）: ラベル情報を直接使うため最も高精度")
print()
print("【重要なポイント】")
print("・対照学習はラベルなしにもかかわらず、教師あり学習に近い性能を達成できる")
print("・実世界ではラベルなしデータの方が圧倒的に多いため、対照学習の価値は大きい")

---

## 8. ピクセル予測 vs 特徴空間予測

### 8.1 World Models における予測

World Models の核心的な問題は、「**次に何が起こるか**」を予測することです。

ここで2つのアプローチがあります：

1. **ピクセル空間予測**: 次の時刻の画像をピクセルレベルで予測する
2. **特徴空間予測**: 次の時刻の画像の「特徴（潜在表現）」を予測する

### 8.2 ピクセル予測の問題

- テクスチャやノイズなど、意味と無関係なディテールも予測する必要がある
- 計算コストが高い（出力次元が大きい）
- 微小なピクセル差が大きな損失を生む

### 8.3 特徴空間予測の利点

- 意味的に重要な変化のみを予測すればよい
- 低次元（例: 128次元）なので計算が軽い
- ノイズに対して頑健

これが **JEPA（Joint Embedding Predictive Architecture）** や **Dreamer** が潜在空間で予測を行う理由です。

In [ ]:
# ============================================================
# 7.6b 埋め込み次元の活性度分析
# ============================================================

# 対照学習で全ての埋め込み次元が使われているか確認
# 「次元崩壊（dimensional collapse）」が起きていないかのチェック

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ランダム初期化の次元別分散
random_var = np.var(random_embeddings, axis=0)
axes[0].bar(range(len(random_var)), np.sort(random_var)[::-1], color='gray', alpha=0.7)
axes[0].set_xlabel('次元（分散の大きい順）', fontsize=11)
axes[0].set_ylabel('分散', fontsize=11)
axes[0].set_title('ランダム初期化: 各次元の分散', fontsize=12)
axes[0].grid(True, alpha=0.3)

# 対照学習後の次元別分散
trained_var = np.var(trained_embeddings, axis=0)
axes[1].bar(range(len(trained_var)), np.sort(trained_var)[::-1], color='dodgerblue', alpha=0.7)
axes[1].set_xlabel('次元（分散の大きい順）', fontsize=11)
axes[1].set_ylabel('分散', fontsize=11)
axes[1].set_title('対照学習後: 各次元の分散', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.suptitle('埋め込み空間の次元活用度', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 有効次元数の推定
random_effective = np.sum(random_var > 0.01 * np.max(random_var))
trained_effective = np.sum(trained_var > 0.01 * np.max(trained_var))

print(f"有効次元数（分散が最大値の1%以上）:")
print(f"  ランダム初期化: {random_effective} / 128")
print(f"  対照学習後:     {trained_effective} / 128")
print()
print("【次元崩壊のチェック】")
print("・全ての次元が情報を持っているか？")
print("・一部の次元に分散が集中していないか？")
print("・次元崩壊が起きると、表現の容量が無駄になる")

In [ ]:
# ============================================================
# 8.4 簡単な予測タスクでの比較実験
# ============================================================

# 疑似的な「次フレーム予測」タスク:
# 同じ画像にノイズを加えたものを「次のフレーム」とみなす
# → ピクセル空間と特徴空間でどちらが頑健に予測できるかを検証

def add_noise_to_image(img, noise_level=0.2):
    """画像にガウシアンノイズを加える（時間変化のシミュレーション）"""
    noise = torch.randn_like(img) * noise_level
    noisy_img = torch.clamp(img + noise, 0.0, 1.0)
    return noisy_img


@torch.no_grad()
def compare_prediction_spaces(model, dataset, noise_levels, device='cpu', n_samples=200):
    """ピクセル空間 vs 特徴空間での予測誤差を比較する

    「元画像→ノイズ画像」の変化において、
    同クラスと異クラスの区別がどの空間でより容易かを測定する。
    """
    model.eval()

    results = {'noise_level': [], 'pixel_same': [], 'pixel_diff': [],
               'feature_same': [], 'feature_diff': []}

    for noise_level in noise_levels:
        pixel_same_dists = []
        pixel_diff_dists = []
        feature_same_dists = []
        feature_diff_dists = []

        # サンプルを取得
        indices = np.random.choice(len(dataset), min(n_samples, len(dataset)), replace=False)

        images = []
        labels = []
        for idx in indices:
            img, label = dataset[idx]
            images.append(img)
            labels.append(label)

        images = torch.stack(images)
        labels = np.array(labels)

        # ノイズを加えた画像を作成
        noisy_images = torch.stack([add_noise_to_image(img, noise_level) for img in images])

        # 特徴量を抽出
        orig_features = model.get_embedding(images.to(device)).cpu()
        noisy_features = model.get_embedding(noisy_images.to(device)).cpu()

        # ペアワイズの距離を計算
        for _ in range(500):
            i, j = np.random.choice(len(images), 2, replace=False)

            # ピクセル空間距離
            pixel_dist = torch.norm(images[i] - noisy_images[j]).item()
            # 特徴空間距離
            feature_dist = torch.norm(orig_features[i] - noisy_features[j]).item()

            if labels[i] == labels[j]:
                pixel_same_dists.append(pixel_dist)
                feature_same_dists.append(feature_dist)
            else:
                pixel_diff_dists.append(pixel_dist)
                feature_diff_dists.append(feature_dist)

        results['noise_level'].append(noise_level)
        results['pixel_same'].append(np.mean(pixel_same_dists))
        results['pixel_diff'].append(np.mean(pixel_diff_dists))
        results['feature_same'].append(np.mean(feature_same_dists))
        results['feature_diff'].append(np.mean(feature_diff_dists))

    return results


print("比較実験を実行中...")
noise_levels = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7]
results = compare_prediction_spaces(
    contrastive_model, test_subset, noise_levels, device=device
)
print("完了！")

In [ ]:
# ============================================================
# 8.5 ピクセル空間 vs 特徴空間での頑健性の比較可視化
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ピクセル空間
axes[0].plot(results['noise_level'], results['pixel_same'], 'o-',
             label='同クラス', color='blue', linewidth=2)
axes[0].plot(results['noise_level'], results['pixel_diff'], 's-',
             label='異クラス', color='red', linewidth=2)
axes[0].set_xlabel('ノイズレベル', fontsize=12)
axes[0].set_ylabel('平均ユークリッド距離', fontsize=12)
axes[0].set_title('ピクセル空間での距離', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# 特徴空間
axes[1].plot(results['noise_level'], results['feature_same'], 'o-',
             label='同クラス', color='blue', linewidth=2)
axes[1].plot(results['noise_level'], results['feature_diff'], 's-',
             label='異クラス', color='red', linewidth=2)
axes[1].set_xlabel('ノイズレベル', fontsize=12)
axes[1].set_ylabel('平均ユークリッド距離', fontsize=12)
axes[1].set_title('特徴空間での距離（対照学習）', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('ノイズに対する頑健性: ピクセル vs 特徴空間', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("・ピクセル空間: ノイズが増えると同クラスも異クラスも距離が増加し、区別が困難に")
print("・特徴空間: ノイズが増えても同クラスと異クラスの距離の差が比較的維持される")
print("→ 特徴空間はノイズに対して頑健 = 意味的な情報を保存している")

In [ ]:
# ============================================================
# 8.6 分離度スコア（separability）の比較
# ============================================================

def separability_score(same_dists, diff_dists):
    """同クラスと異クラスの距離の分離度を計算する

    値が大きいほど、同クラスと異クラスの区別がしやすい。
    """
    numerator = abs(np.mean(diff_dists) - np.mean(same_dists))
    denominator = np.std(diff_dists) + np.std(same_dists) + 1e-8
    return numerator / denominator

pixel_seps = []
feature_seps = []

for i, nl in enumerate(results['noise_level']):
    # 分離度の近似計算（平均値の比で代用）
    pixel_sep = results['pixel_diff'][i] / (results['pixel_same'][i] + 1e-8)
    feature_sep = results['feature_diff'][i] / (results['feature_same'][i] + 1e-8)
    pixel_seps.append(pixel_sep)
    feature_seps.append(feature_sep)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(results['noise_level'], pixel_seps, 'o-', label='ピクセル空間',
        color='gray', linewidth=2, markersize=8)
ax.plot(results['noise_level'], feature_seps, 's-', label='特徴空間（対照学習）',
        color='dodgerblue', linewidth=2, markersize=8)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='分離不可能ライン')
ax.set_xlabel('ノイズレベル', fontsize=12)
ax.set_ylabel('距離比（異クラス/同クラス）', fontsize=12)
ax.set_title('ノイズ増加に対する分離度の変化', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("【結論】")
print("・特徴空間の距離比は、ノイズが増えても高い値を維持する傾向がある")
print("・ピクセル空間の距離比は、ノイズに対して急速に1.0（分離不可能）に近づく")
print()
print("→ World Models が潜在空間で予測を行う根拠がここにある：")
print("  ピクセルの微小な変化（ノイズ、照明変化など）に影響されず、")
print("  意味的に重要な変化のみを予測できる")

In [ ]:
# ============================================================
# 8.3b 簡易予測器の比較: ピクセル vs 特徴空間
# ============================================================

# 簡単な線形予測器をピクセル空間と特徴空間で訓練し、
# 「次の画像」（同じ画像にノイズを加えたもの）の予測精度を比較する

@torch.no_grad()
def measure_prediction_error(model, dataset, noise_level=0.2, device='cpu', n_samples=200):
    """ピクセル空間と特徴空間での予測MSEを比較する"""
    model.eval()

    pixel_mse_list = []
    feature_mse_list = []

    for i in range(min(n_samples, len(dataset))):
        img, _ = dataset[i]
        noisy = add_noise_to_image(img, noise_level)

        # ピクセル空間のMSE（元画像で「予測」した場合の誤差）
        pixel_mse = F.mse_loss(img, noisy).item()
        pixel_mse_list.append(pixel_mse)

        # 特徴空間のMSE
        feat_orig = model.get_embedding(img.unsqueeze(0).to(device)).cpu()
        feat_noisy = model.get_embedding(noisy.unsqueeze(0).to(device)).cpu()
        feature_mse = F.mse_loss(feat_orig, feat_noisy).item()
        feature_mse_list.append(feature_mse)

    return np.mean(pixel_mse_list), np.mean(feature_mse_list)

noise_test_levels = [0.05, 0.1, 0.2, 0.3, 0.5]
pixel_errors = []
feature_errors = []

for nl in noise_test_levels:
    p_err, f_err = measure_prediction_error(
        contrastive_model, test_subset, noise_level=nl, device=device
    )
    pixel_errors.append(p_err)
    feature_errors.append(f_err)

# 正規化して比較しやすくする
pixel_errors_norm = np.array(pixel_errors) / pixel_errors[0]
feature_errors_norm = np.array(feature_errors) / (feature_errors[0] + 1e-8)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(noise_test_levels, pixel_errors_norm, 'o-', label='ピクセル空間 MSE（正規化）',
        color='gray', linewidth=2, markersize=8)
ax.plot(noise_test_levels, feature_errors_norm, 's-', label='特徴空間 MSE（正規化）',
        color='dodgerblue', linewidth=2, markersize=8)
ax.set_xlabel('ノイズレベル', fontsize=12)
ax.set_ylabel('正規化 MSE（初期値=1）', fontsize=12)
ax.set_title('ノイズに対する予測誤差の増加速度', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("・ピクセル空間のMSEはノイズレベルに比例して増加（ノイズ分がそのまま誤差に）")
print("・特徴空間のMSEはノイズに対してより緩やかに増加する傾向がある")
print("→ 特徴空間での予測は、入力のノイズに対してより頑健")

In [ ]:
# ============================================================
# 8.7 ノイズ画像の可視化例
# ============================================================

fig, axes = plt.subplots(2, 6, figsize=(14, 5))

# 2つのサンプル画像
sample_imgs = []
sample_labels = []
for idx in [0, 50]:
    img, label = test_subset[idx]
    sample_imgs.append(img)
    sample_labels.append(label)

for row, (img, label) in enumerate(zip(sample_imgs, sample_labels)):
    for col, nl in enumerate([0.0, 0.1, 0.2, 0.3, 0.5, 0.7]):
        noisy = add_noise_to_image(img, nl)
        axes[row, col].imshow(noisy.squeeze(), cmap='gray')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f'noise={nl}', fontsize=10)
    axes[row, 0].set_ylabel(class_names[label], fontsize=11, rotation=0, labelpad=50)

plt.suptitle('ノイズレベルの変化例', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("人間にはノイズがあっても何のカテゴリかは分かる")
print("→ 良い表現学習は、この『人間的な認識の頑健さ』を獲得する")

### 8.8 World Models との接続まとめ

本セクションの実験から得られた知見を World Models の文脈でまとめます：

| アプローチ | 代表モデル | 予測空間 | 利点 | 欠点 |
|------------|-----------|----------|------|------|
| ピクセル予測 | 初期のVAE-based WM | ピクセル | 直接的で理解しやすい | ノイズ、計算コスト |
| 潜在空間予測 | Dreamer, PlaNet | 学習された潜在空間 | 効率的、ノイズ頑健 | 表現の質に依存 |
| JEPA方式 | V-JEPA, I-JEPA | 共同埋め込み空間 | コラプス回避が容易 | 設計がやや複雑 |

**核心的なメッセージ**: 良い表現を学習できれば、その表現空間での予測はピクセル空間での予測よりも効率的で頑健です。これが「表現学習 + 潜在空間予測」という World Models のパラダイムの根拠です。

---

## 9. まとめ・よくある間違い・確認クイズ

### 9.1 このノートブックで学んだこと

1. **ピクセル空間の限界**: ピクセル距離は意味的な類似度と一致しない。同じカテゴリの画像でもピクセル的には大きく異なりうる

2. **対照学習（InfoNCE）**: ラベルなしで、データ拡張を活用して意味的に有用な表現を学習できる。正例ペア（同じ画像の異なる拡張）を近づけ、負例ペア（異なる画像）を遠ざける

3. **t-SNE 可視化**: 高次元の埋め込みを2次元で可視化し、学習された表現がクラス構造を捉えているかを確認できる

4. **線形プローブ評価**: エンコーダの重みを凍結し、線形分類器のみで評価することで、表現の質を定量的に測定できる

5. **特徴空間予測の頑健性**: ノイズに対して、特徴空間はピクセル空間より頑健に同クラス/異クラスの区別を維持する。これが World Models が潜在空間予測を採用する理由

### 9.2 よくある間違い（3つ）

#### 間違い 1: プロジェクションヘッドの出力で評価してしまう

**誤り**: 対照学習後、プロジェクションヘッドの出力を下流タスクの特徴として使う

**正しい**: プロジェクションヘッドは **学習時の損失計算用** であり、評価時は **エンコーダの埋め込み** を使う。SimCLR の論文で、プロジェクションヘッドの出力よりエンコーダの出力の方が下流タスクの精度が高いことが示されている

#### 間違い 2: 温度パラメータを無視する

**誤り**: InfoNCE の温度 $\tau$ を 1.0 のまま放置する

**正しい**: $\tau$ は学習の安定性と性能に大きな影響を与える。小さすぎると勾配が不安定になり、大きすぎると識別力が低下する。一般に 0.07〜0.5 の範囲で調整する

#### 間違い 3: データ拡張が弱すぎる/強すぎる

**誤り**: 拡張をほとんど適用しない、または元画像が判別不能なほど強い拡張を適用する

**正しい**: 適切な強度のデータ拡張が重要。弱すぎるとモデルがショートカット（例: 色のみで判定）を学習し、強すぎると意味的情報自体が失われる

### 9.3 確認クイズ（5問）

---

**Q1**: 対照学習でデータ拡張を使って正例ペアを作る理由は何ですか？

<details>
<summary>解答を表示</summary>

ラベルがない状況で「意味的に同じ」サンプルのペアを作るためです。同じ画像に異なるデータ拡張を適用しても意味（カテゴリ）は変わらないため、これらを正例ペアとして扱い、表現空間で近くなるように学習します。
</details>

---

**Q2**: InfoNCE 損失における温度パラメータ $\tau$ を小さくすると何が起こりますか？

<details>
<summary>解答を表示</summary>

ソフトマックスの確率分布が鋭くなります。つまり、正例と負例の区別がより厳格になり、最も類似度の高いペアに確率が集中します。小さすぎると勾配が不安定になるリスクがあります。
</details>

---

**Q3**: 線形プローブ評価でエンコーダの重みを凍結する理由は何ですか？

<details>
<summary>解答を表示</summary>

表現（エンコーダの出力）の質そのものを評価するためです。もしエンコーダの重みも更新してしまうと、線形分類器の学習中に表現自体が変化してしまい、「元の表現がどれほど良かったか」を正しく測定できなくなります。
</details>

---

**Q4**: World Models（例: Dreamer, JEPA）がピクセル空間ではなく潜在空間で予測を行う主な利点を2つ挙げてください。

<details>
<summary>解答を表示</summary>

1. **ノイズに対する頑健性**: 潜在空間はピクセルの微小な変化（ノイズ、照明変化など）に影響されにくく、意味的に重要な変化のみを捉える。
2. **計算効率**: 低次元の潜在空間（例: 128次元）での予測は、高次元のピクセル空間（例: 数万次元）での予測より計算コストが大幅に低い。
</details>

---

**Q5**: 対照学習の訓練後、プロジェクションヘッドの出力ではなくエンコーダの埋め込みを下流タスクの特徴として使うべき理由は何ですか？

<details>
<summary>解答を表示</summary>

プロジェクションヘッドは対照学習の損失関数に特化した変換であり、タスク固有の情報を捨てている可能性があります。SimCLR の研究で、エンコーダの埋め込みの方がより汎用的な情報を保持しており、様々な下流タスクで高い性能を示すことが実験的に確認されています。
</details>

---

### 次のステップ

このノートブックでは表現学習の基礎と、それが予測タスクにとってなぜ重要かを学びました。次のステップとして：

- **潜在空間ダイナミクスモデル**: 学習した表現空間で状態遷移を予測するモデル（Dreamer の世界モデル部分）
- **JEPA アーキテクチャ**: エネルギーベースのアプローチによる、コラプスしない予測学習
- **行動条件付き予測**: エージェントの行動を条件として潜在空間の次状態を予測する

---

### 参考文献

1. Chen, T. et al. (2020). *A Simple Framework for Contrastive Learning of Visual Representations (SimCLR)*. ICML.
2. Oord, A. et al. (2018). *Representation Learning with Contrastive Predictive Coding*. arXiv:1807.03748.
3. LeCun, Y. (2022). *A Path Towards Autonomous Machine Intelligence*. openreview.net. (JEPAの提案)
4. Hafner, D. et al. (2023). *Mastering Diverse Domains through World Models (DreamerV3)*. arXiv.